In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import shutil, os
shutil.copytree("/content/drive/MyDrive/IITD_Palmprint", "/content/IITD_Palmprint")

'/content/IITD_Palmprint'

In [3]:
import tensorflow as tf
import matplotlib.pyplot as plt
import pickle
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Dropout, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

In [4]:
train_dir = "/content/IITD_Palmprint"
img_size = 224
batch_size = 32

In [5]:
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2)

In [6]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True)

validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False)

Found 4162 images belonging to 3 classes.
Found 1039 images belonging to 3 classes.


In [7]:
num_classes = train_generator.num_classes
print("Classes:", num_classes)
with open("class_names.pkl", "wb") as f:
  pickle.dump(train_generator.class_indices, f)

Classes: 3


In [8]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(img_size, img_size, 3)),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64,(3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128,(3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(256,(3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Flatten(),

    Dense(256, activation='relu', name='feature_layer'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 24, 24, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_layer (Dense)           │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,828,547 (37.49 MB)

 Trainable params: 9,827,587 (37.49 MB)

 Non-trainable params: 960 (3.75 KB)

In [10]:
epochs = 40
callbacks = [
    ModelCheckpoint("best_palm_model.h5", monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6),
    EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True)]

history = model.fit(train_generator, validation_data=validation_generator, epochs=epochs, callbacks=callbacks)


Epoch 1/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 796ms/step - accuracy: 0.8223 - loss: 0.7142
Epoch 1: val_accuracy improved from None to 0.24928, saving model to best_palm_model.h5



Epoch 1: finished saving model to best_palm_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 148s 1s/step - accuracy: 0.8888 - loss: 0.3741 - val_accuracy: 0.2493 - val_loss: 6.9661 - learning_rate: 1.0000e-04
Epoch 2/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 730ms/step - accuracy: 0.9575 - loss: 0.1082
Epoch 2: val_accuracy improved from 0.24928 to 0.74976, saving model to best_palm_model.h5



Epoch 2: finished saving model to best_palm_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 119s 911ms/step - accuracy: 0.9647 - loss: 0.0887 - val_accuracy: 0.7498 - val_loss: 1.0222 - learning_rate: 1.0000e-04
Epoch 3/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 754ms/step - accuracy: 0.9740 - loss: 0.0710
Epoch 3: val_accuracy did not improve from 0.74976
131/131 ━━━━━━━━━━━━━━━━━━━━ 120s 920ms/step - accuracy: 0.9688 - loss: 0.0807 - val_accuracy: 0.5457 - val_loss: 1.5205 - learning_rate: 1.0000e-04
Epoch 4/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 733ms/step - accuracy: 0.9890 - loss: 0.0356
Epoch 4: val_accuracy improved from 0.74976 to 0.90375, saving model to best_palm_model.h5



Epoch 4: finished saving model to best_palm_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 119s 907ms/step - accuracy: 0.9868 - loss: 0.0385 - val_accuracy: 0.9038 - val_loss: 0.2247 - learning_rate: 1.0000e-04
Epoch 5/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 731ms/step - accuracy: 0.9812 - loss: 0.0423
Epoch 5: val_accuracy did not improve from 0.90375
131/131 ━━━━━━━━━━━━━━━━━━━━ 137s 1s/step - accuracy: 0.9813 - loss: 0.0450 - val_accuracy: 0.9018 - val_loss: 0.2378 - learning_rate: 1.0000e-04
Epoch 6/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 723ms/step - accuracy: 0.9904 - loss: 0.0284
Epoch 6: val_accuracy improved from 0.90375 to 0.93167, saving model to best_palm_model.h5



Epoch 6: finished saving model to best_palm_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 118s 899ms/step - accuracy: 0.9899 - loss: 0.0269 - val_accuracy: 0.9317 - val_loss: 0.1551 - learning_rate: 1.0000e-04
Epoch 7/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 747ms/step - accuracy: 0.9941 - loss: 0.0173
Epoch 7: val_accuracy did not improve from 0.93167
131/131 ━━━━━━━━━━━━━━━━━━━━ 119s 913ms/step - accuracy: 0.9945 - loss: 0.0154 - val_accuracy: 0.9307 - val_loss: 0.1625 - learning_rate: 1.0000e-04
Epoch 8/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 750ms/step - accuracy: 0.9945 - loss: 0.0164
Epoch 8: val_accuracy improved from 0.93167 to 0.93744, saving model to best_palm_model.h5



Epoch 8: finished saving model to best_palm_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 120s 919ms/step - accuracy: 0.9940 - loss: 0.0179 - val_accuracy: 0.9374 - val_loss: 0.1637 - learning_rate: 1.0000e-04
Epoch 9/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 730ms/step - accuracy: 0.9940 - loss: 0.0126
Epoch 9: val_accuracy improved from 0.93744 to 0.94033, saving model to best_palm_model.h5



Epoch 9: finished saving model to best_palm_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 139s 896ms/step - accuracy: 0.9954 - loss: 0.0109 - val_accuracy: 0.9403 - val_loss: 0.1548 - learning_rate: 5.0000e-05
Epoch 10/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 712ms/step - accuracy: 0.9962 - loss: 0.0093
Epoch 10: val_accuracy did not improve from 0.94033
131/131 ━━━━━━━━━━━━━━━━━━━━ 114s 871ms/step - accuracy: 0.9976 - loss: 0.0079 - val_accuracy: 0.9249 - val_loss: 0.1816 - learning_rate: 5.0000e-05
Epoch 11/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 720ms/step - accuracy: 0.9943 - loss: 0.0185
Epoch 11: val_accuracy improved from 0.94033 to 0.94225, saving model to best_palm_model.h5



Epoch 11: finished saving model to best_palm_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 116s 888ms/step - accuracy: 0.9971 - loss: 0.0107 - val_accuracy: 0.9423 - val_loss: 0.1368 - learning_rate: 5.0000e-05
Epoch 12/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 718ms/step - accuracy: 0.9957 - loss: 0.0135
Epoch 12: val_accuracy did not improve from 0.94225
131/131 ━━━━━━━━━━━━━━━━━━━━ 115s 879ms/step - accuracy: 0.9974 - loss: 0.0091 - val_accuracy: 0.9423 - val_loss: 0.1315 - learning_rate: 5.0000e-05
Epoch 13/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 721ms/step - accuracy: 0.9979 - loss: 0.0072
Epoch 13: val_accuracy improved from 0.94225 to 0.94899, saving model to best_palm_model.h5



Epoch 13: finished saving model to best_palm_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 116s 885ms/step - accuracy: 0.9976 - loss: 0.0073 - val_accuracy: 0.9490 - val_loss: 0.1303 - learning_rate: 5.0000e-05
Epoch 14/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 736ms/step - accuracy: 0.9984 - loss: 0.0066
Epoch 14: val_accuracy improved from 0.94899 to 0.95765, saving model to best_palm_model.h5



Epoch 14: finished saving model to best_palm_model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 118s 904ms/step - accuracy: 0.9976 - loss: 0.0074 - val_accuracy: 0.9577 - val_loss: 0.1205 - learning_rate: 5.0000e-05
Epoch 15/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 728ms/step - accuracy: 0.9982 - loss: 0.0054
Epoch 15: val_accuracy did not improve from 0.95765
131/131 ━━━━━━━━━━━━━━━━━━━━ 116s 890ms/step - accuracy: 0.9981 - loss: 0.0062 - val_accuracy: 0.9355 - val_loss: 0.1574 - learning_rate: 5.0000e-05
Epoch 16/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 716ms/step - accuracy: 0.9968 - loss: 0.0113
Epoch 16: val_accuracy did not improve from 0.95765
131/131 ━━━━━━━━━━━━━━━━━━━━ 115s 879ms/step - accuracy: 0.9969 - loss: 0.0107 - val_accuracy: 0.9538 - val_loss: 0.1172 - learning_rate: 5.0000e-05
Epoch 17/40
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 736ms/step - accuracy: 0.9983 - loss: 0.0056
Epoch 17: val_accuracy did not improve from 0.95765
131/131 ━━━━━━━━━━━━━━━━━━━━ 137s 1s/step - accuracy: 0.9978 - loss: 0.0060

In [11]:
model.save("final_palm_model.h5")
print("Training Completed!")

Training Completed!
